In [8]:
# ==========================================================
# Import Required Libraries
# These libraries are used for data preparation, encoding,scaling, dataset splitting, and saving preprocessing files.
# ==========================================================

import os
import re
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
# ==========================================================
# Load and Recreate the Cleaned Dataset
# This step loads the original Telco Customer Churn dataset, converts TotalCharges into a numerical data type, and removes the incomplete customer records found in Phase 2.
# ==========================================================

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df = df.dropna().reset_index(drop=True)

print("Dataset loaded and cleaned successfully.")
print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())

df.head()

Dataset loaded and cleaned successfully.
Dataset Shape: (7032, 21)
Total Missing Values: 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
# ==========================================================
# Define Dataset Columns
# This step identifies the target variable, ID column,numerical features, binary numerical features, and categorical features required for preprocessing.
# ==========================================================

target_column = "Churn"
id_column = "customerID"

numerical_features = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

continuous_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

binary_numeric_features = [
    "SeniorCitizen"
]

categorical_features = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

print("Target Variable:", target_column)
print("ID Column:", id_column)

print("\nNumerical Features:")
print(numerical_features)

print("\nContinuous Features to Scale:")
print(continuous_features)

print("\nBinary Numerical Features:")
print(binary_numeric_features)

print("\nCategorical Features:")
print(categorical_features)

Target Variable: Churn
ID Column: customerID

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Continuous Features to Scale:
['tenure', 'MonthlyCharges', 'TotalCharges']

Binary Numerical Features:
['SeniorCitizen']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [11]:
# ==========================================================
# Verify the Selected Features
# This step confirms that every selected feature exists in the dataset and that no required column is missing.
# ==========================================================

selected_columns = (
    [id_column]
    + numerical_features
    + categorical_features
    + [target_column]
)

missing_columns = [
    column for column in selected_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"The following columns are missing: {missing_columns}"
    )

if len(selected_columns) != len(set(selected_columns)):
    raise ValueError(
        "Duplicate columns were found in the feature lists."
    )

print("All selected columns were found in the dataset.")
print("Number of Input Features:", len(numerical_features) + len(categorical_features))
print("Number of Numerical Features:", len(numerical_features))
print("Number of Categorical Features:", len(categorical_features))

All selected columns were found in the dataset.
Number of Input Features: 19
Number of Numerical Features: 4
Number of Categorical Features: 15


In [12]:
# ==========================================================
# Separate ID, Input Features, and Target customerID is preserved for later prediction tracking but is removed from the machine learning input features.
# ==========================================================

customer_ids = df[id_column].copy()

X = df[
    numerical_features + categorical_features
].copy()

y_original = df[target_column].copy()

print("Input Feature Shape:", X.shape)
print("Target Shape:", y_original.shape)
print("Customer ID Shape:", customer_ids.shape)

print("\nColumns Used for Model Input:")
print(X.columns.tolist())

Input Feature Shape: (7032, 19)
Target Shape: (7032,)
Customer ID Shape: (7032,)

Columns Used for Model Input:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [13]:
# ==========================================================
# Encode the Target Variable
# This step converts Churn from Yes and No values into:
# No = 0
# Yes = 1
# ==========================================================

target_mapping = {
    "No": 0,
    "Yes": 1
}

y = y_original.map(target_mapping)

if y.isnull().any():
    invalid_values = y_original[y.isnull()].unique()

    raise ValueError(
        f"Unexpected target values found: {invalid_values}"
    )

y = y.astype(int)
y.name = "Churn"

print("Target variable encoded successfully.")

print("\nTarget Mapping:")
print(target_mapping)

print("\nEncoded Target Distribution:")
print(y.value_counts())

print("\nEncoded Target Percentage:")
print((y.value_counts(normalize=True) * 100).round(2))

Target variable encoded successfully.

Target Mapping:
{'No': 0, 'Yes': 1}

Encoded Target Distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64

Encoded Target Percentage:
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64


In [14]:
# ==========================================================
# Split the Dataset into Training and Testing Sets
# An 80-20 split is used. Stratification preserves the churn distribution in both the training and testing datasets.
# ==========================================================

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X,
    y,
    customer_ids,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train-test split completed successfully.")

print("\nTraining Feature Shape:", X_train.shape)
print("Testing Feature Shape:", X_test.shape)
print("Training Target Shape:", y_train.shape)
print("Testing Target Shape:", y_test.shape)

print("\nTraining Target Percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting Target Percentage:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Train-test split completed successfully.

Training Feature Shape: (5625, 19)
Testing Feature Shape: (1407, 19)
Training Target Shape: (5625,)
Testing Target Shape: (1407,)

Training Target Percentage:
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64

Testing Target Percentage:
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64


In [15]:
# ==========================================================
# Create the Preprocessing Transformations
# Continuous numerical features are standardized.
# SeniorCitizen remains in its original binary 0 and 1 form.
# Categorical variables are converted using one-hot encoding.
# ==========================================================

try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_features
        ),
        (
            "binary",
            "passthrough",
            binary_numeric_features
        ),
        (
            "categorical",
            one_hot_encoder,
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

print("Preprocessing transformations created successfully.")
print(preprocessor)

Preprocessing transformations created successfully.
ColumnTransformer(transformers=[('continuous', StandardScaler(),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges']),
                                ('binary', 'passthrough', ['SeniorCitizen']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])


In [17]:
# ==========================================================
# Fit and Apply the Preprocessing Transformations
# The preprocessor is fitted only on the training data to prevent information from the testing data leaking into the machine learning training process.
# ==========================================================

X_train_processed_array = preprocessor.fit_transform(X_train)

X_test_processed_array = preprocessor.transform(X_test)

print("Preprocessing completed successfully.")

print(
    "Processed Training Shape:",
    X_train_processed_array.shape
)

print(
    "Processed Testing Shape:",
    X_test_processed_array.shape
)

Preprocessing completed successfully.
Processed Training Shape: (5625, 45)
Processed Testing Shape: (1407, 45)


In [18]:
# ==========================================================
# Create Readable Processed Feature Names
# This step retrieves the generated feature names and removes technical prefixes and unsuitable special characters.
# ==========================================================

raw_feature_names = preprocessor.get_feature_names_out()

def clean_feature_name(feature_name):
    feature_name = feature_name.split("__", 1)[-1]

    feature_name = re.sub(
        r"[^A-Za-z0-9_]+",
        "_",
        feature_name
    )

    feature_name = re.sub(
        r"_+",
        "_",
        feature_name
    )

    return feature_name.strip("_")

processed_feature_names = [
    clean_feature_name(name)
    for name in raw_feature_names
]

if len(processed_feature_names) != len(set(processed_feature_names)):
    raise ValueError(
        "Duplicate processed feature names were created."
    )

print("Number of Processed Features:", len(processed_feature_names))

print("\nProcessed Feature Names:")
for feature in processed_feature_names:
    print(feature)

Number of Processed Features: 45

Processed Feature Names:
tenure
MonthlyCharges
TotalCharges
SeniorCitizen
gender_Female
gender_Male
Partner_No
Partner_Yes
Dependents_No
Dependents_Yes
PhoneService_No
PhoneService_Yes
MultipleLines_No
MultipleLines_No_phone_service
MultipleLines_Yes
InternetService_DSL
InternetService_Fiber_optic
InternetService_No
OnlineSecurity_No
OnlineSecurity_No_internet_service
OnlineSecurity_Yes
OnlineBackup_No
OnlineBackup_No_internet_service
OnlineBackup_Yes
DeviceProtection_No
DeviceProtection_No_internet_service
DeviceProtection_Yes
TechSupport_No
TechSupport_No_internet_service
TechSupport_Yes
StreamingTV_No
StreamingTV_No_internet_service
StreamingTV_Yes
StreamingMovies_No
StreamingMovies_No_internet_service
StreamingMovies_Yes
Contract_Month_to_month
Contract_One_year
Contract_Two_year
PaperlessBilling_No
PaperlessBilling_Yes
PaymentMethod_Bank_transfer_automatic
PaymentMethod_Credit_card_automatic
PaymentMethod_Electronic_check
PaymentMethod_Mailed_chec

In [19]:
# ==========================================================
# Convert Processed Data into Pandas DataFrames
# DataFrames make the processed training and testing data easier to inspect, save, and use in the next notebook.
# ==========================================================

X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=processed_feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=processed_feature_names
)

y_train_processed = y_train.reset_index(drop=True).to_frame()
y_test_processed = y_test.reset_index(drop=True).to_frame()

print("Processed DataFrames created successfully.")

print("\nProcessed Training Features:")
print(X_train_processed.shape)

print("\nProcessed Testing Features:")
print(X_test_processed.shape)

print("\nTraining Target:")
print(y_train_processed.shape)

print("\nTesting Target:")
print(y_test_processed.shape)

X_train_processed.head()

Processed DataFrames created successfully.

Processed Training Features:
(5625, 45)

Processed Testing Features:
(1407, 45)

Training Target:
(5625, 1)

Testing Target:
(1407, 1)


,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,StreamingMovies_Yes,Contract_Month_to_month,Contract_One_year,Contract_Two_year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank_transfer_automatic,PaymentMethod_Credit_card_automatic,PaymentMethod_Electronic_check,PaymentMethod_Mailed_check
0,1.321816,0.981556,1.659900,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-0.267410,-0.971546,-0.562252,0.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1.444064,0.837066,1.756104,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-1.204646,0.641092,-0.908326,0.0,0.0,1.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.669826,-0.808787,-0.101561,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [21]:
# ==========================================================
# Validate the Processed Datasets
# This step checks that the processed data contains no missing values, infinite values, or inconsistent columns.
# ==========================================================

train_missing_values = X_train_processed.isnull().sum().sum()
test_missing_values = X_test_processed.isnull().sum().sum()

train_infinite_values = np.isinf(
    X_train_processed.to_numpy()
).sum()

test_infinite_values = np.isinf(
    X_test_processed.to_numpy()
).sum()

same_columns = (
    X_train_processed.columns.tolist()
    == X_test_processed.columns.tolist()
)

if train_missing_values != 0:
    raise ValueError(
        "Missing values were found in the processed training data."
    )

if test_missing_values != 0:
    raise ValueError(
        "Missing values were found in the processed testing data."
    )

if train_infinite_values != 0:
    raise ValueError(
        "Infinite values were found in the processed training data."
    )

if test_infinite_values != 0:
    raise ValueError(
        "Infinite values were found in the processed testing data."
    )

if not same_columns:
    raise ValueError(
        "Training and testing feature columns do not match."
    )

print("Processed dataset validation completed successfully.")

print("\nTraining Missing Values:", train_missing_values)
print("Testing Missing Values:", test_missing_values)

print("Training Infinite Values:", train_infinite_values)
print("Testing Infinite Values:", test_infinite_values)

print("Training and Testing Columns Match:", same_columns)

print(
    "All Processed Values Are Numeric:",
    all(
        pd.api.types.is_numeric_dtype(dtype)
        for dtype in X_train_processed.dtypes
    )
)

Processed dataset validation completed successfully.

Training Missing Values: 0
Testing Missing Values: 0
Training Infinite Values: 0
Testing Infinite Values: 0
Training and Testing Columns Match: True
All Processed Values Are Numeric: True


In [23]:
# ==========================================================
# Inspect the Preprocessing Results
# This step displays a small sample of the transformed data and confirms how the continuous features were scaled.
# ==========================================================

print("Processed Training Data Sample:")

display(
    X_train_processed.head()
)

print("\nScaled Continuous Feature Statistics:")

display(
    X_train_processed[
        continuous_features
    ].describe().round(3)
)

Processed Training Data Sample:


,tenure,MonthlyCharges,TotalCharges,SeniorCitizen,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,StreamingMovies_Yes,Contract_Month_to_month,Contract_One_year,Contract_Two_year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank_transfer_automatic,PaymentMethod_Credit_card_automatic,PaymentMethod_Electronic_check,PaymentMethod_Mailed_check
0,1.321816,0.981556,1.659900,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-0.267410,-0.971546,-0.562252,0.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1.444064,0.837066,1.756104,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-1.204646,0.641092,-0.908326,0.0,0.0,1.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.669826,-0.808787,-0.101561,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0



Scaled Continuous Feature Statistics:


,tenure,MonthlyCharges,TotalCharges
count,5625.000,5625.000,5625.000
mean,-0.000,0.000,0.000
std,1.000,1.000,1.000
min,-1.286,-1.548,-1.003
25%,-0.960,-0.970,-0.830
50%,-0.145,0.186,-0.392
75%,0.955,0.832,0.659
max,1.607,1.782,2.805


In [24]:
# ==========================================================
# Create Training and Testing Reference Data
# These files preserve the original customer information and customer IDs for later model predictions, SHAP explanations,and Power BI integration.
# ==========================================================

train_reference = df.loc[
    X_train.index
].copy().reset_index(drop=True)

test_reference = df.loc[
    X_test.index
].copy().reset_index(drop=True)

train_reference["ChurnEncoded"] = (
    y_train.reset_index(drop=True)
)

test_reference["ChurnEncoded"] = (
    y_test.reset_index(drop=True)
)

print("Reference datasets created successfully.")

print("\nTraining Reference Shape:", train_reference.shape)
print("Testing Reference Shape:", test_reference.shape)

print("\nTraining Customer IDs Match:", (
    train_reference[id_column].tolist()
    == id_train.reset_index(drop=True).tolist()
))

print("Testing Customer IDs Match:", (
    test_reference[id_column].tolist()
    == id_test.reset_index(drop=True).tolist()
))

Reference datasets created successfully.

Training Reference Shape: (5625, 22)
Testing Reference Shape: (1407, 22)

Training Customer IDs Match: True
Testing Customer IDs Match: True


In [26]:
# ==========================================================
# Create the Feature Name Mapping
# This file connects the original preprocessing output names with the cleaner names used in the processed datasets.
# ==========================================================

feature_name_mapping = pd.DataFrame({
    "RawFeatureName": raw_feature_names,
    "ProcessedFeatureName": processed_feature_names
})

print("Feature name mapping created successfully.")

feature_name_mapping.head(10)

Feature name mapping created successfully.


,RawFeatureName,ProcessedFeatureName
0,continuous__tenure,tenure
1,continuous__MonthlyCharges,MonthlyCharges
2,continuous__TotalCharges,TotalCharges
3,binary__SeniorCitizen,SeniorCitizen
4,categorical__gender_Female,gender_Female
5,categorical__gender_Male,gender_Male
6,categorical__Partner_No,Partner_No
7,categorical__Partner_Yes,Partner_Yes
8,categorical__Dependents_No,Dependents_No
9,categorical__Dependents_Yes,Dependents_Yes


In [27]:
# ==========================================================
# Save the Processed Data and Preprocessing Files
# These outputs will be used in the machine learning, explainability, and Power BI phases of the thesis.
# ==========================================================

output_directory = "phase3_outputs"

os.makedirs(
    output_directory,
    exist_ok=True
)

df.to_csv(
    os.path.join(
        output_directory,
        "cleaned_telco_churn.csv"
    ),
    index=False
)

X_train_processed.to_csv(
    os.path.join(
        output_directory,
        "X_train_processed.csv"
    ),
    index=False
)

X_test_processed.to_csv(
    os.path.join(
        output_directory,
        "X_test_processed.csv"
    ),
    index=False
)

y_train_processed.to_csv(
    os.path.join(
        output_directory,
        "y_train.csv"
    ),
    index=False
)

y_test_processed.to_csv(
    os.path.join(
        output_directory,
        "y_test.csv"
    ),
    index=False
)

train_reference.to_csv(
    os.path.join(
        output_directory,
        "train_reference.csv"
    ),
    index=False
)

test_reference.to_csv(
    os.path.join(
        output_directory,
        "test_reference.csv"
    ),
    index=False
)

feature_name_mapping.to_csv(
    os.path.join(
        output_directory,
        "feature_name_mapping.csv"
    ),
    index=False
)

joblib.dump(
    preprocessor,
    os.path.join(
        output_directory,
        "preprocessor.joblib"
    )
)

print("Phase 3 data files saved successfully.")

Phase 3 data files saved successfully.


In [28]:
# ==========================================================
# Save Preprocessing Metadata
# This file records the feature lists, target mapping, dataset split, and preprocessing decisions required for reproducibility in the following thesis phases.
# ==========================================================

preprocessing_metadata = {
    "dataset_name": "IBM Telco Customer Churn",
    "original_rows": 7043,
    "cleaned_rows": int(len(df)),
    "removed_rows": int(7043 - len(df)),
    "id_column": id_column,
    "target_column": target_column,
    "target_mapping": target_mapping,
    "numerical_features": numerical_features,
    "continuous_scaled_features": continuous_features,
    "binary_numeric_features": binary_numeric_features,
    "categorical_features": categorical_features,
    "original_input_feature_count": int(X.shape[1]),
    "processed_feature_count": int(X_train_processed.shape[1]),
    "training_rows": int(X_train_processed.shape[0]),
    "testing_rows": int(X_test_processed.shape[0]),
    "test_size": 0.20,
    "random_state": 42,
    "stratified_split": True,
    "continuous_transformation": "StandardScaler",
    "categorical_transformation": "OneHotEncoder",
    "unknown_category_handling": "ignore",
    "senior_citizen_transformation": "passthrough"
}

with open(
    os.path.join(
        output_directory,
        "preprocessing_metadata.json"
    ),
    "w",
    encoding="utf-8"
) as metadata_file:
    json.dump(
        preprocessing_metadata,
        metadata_file,
        indent=4
    )

print("Preprocessing metadata saved successfully.")

print("\nMetadata Summary:")
print(json.dumps(preprocessing_metadata, indent=4))

Preprocessing metadata saved successfully.

Metadata Summary:
{
    "dataset_name": "IBM Telco Customer Churn",
    "original_rows": 7043,
    "cleaned_rows": 7032,
    "removed_rows": 11,
    "id_column": "customerID",
    "target_column": "Churn",
    "target_mapping": {
        "No": 0,
        "Yes": 1
    },
    "numerical_features": [
        "SeniorCitizen",
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ],
    "continuous_scaled_features": [
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ],
    "binary_numeric_features": [
        "SeniorCitizen"
    ],
    "categorical_features": [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMeth

In [29]:
# ==========================================================
# Confirm the Saved Phase 3 Files
# This step lists every file created during preprocessing and displays its file size.
# ==========================================================

saved_files = sorted(
    os.listdir(output_directory)
)

print("Phase 3 Output Files:\n")

for file_name in saved_files:
    file_path = os.path.join(
        output_directory,
        file_name
    )

    file_size_kb = os.path.getsize(
        file_path
    ) / 1024

    print(
        f"{file_name} - {file_size_kb:.2f} KB"
    )

print("\nTotal Files Saved:", len(saved_files))

Phase 3 Output Files:

X_test_processed.csv - 312.45 KB
X_train_processed.csv - 1246.38 KB
cleaned_telco_churn.csv - 947.43 KB
feature_name_mapping.csv - 2.44 KB
preprocessing_metadata.json - 1.32 KB
preprocessor.joblib - 6.90 KB
test_reference.csv - 193.03 KB
train_reference.csv - 768.41 KB
y_test.csv - 2.75 KB
y_train.csv - 10.99 KB

Total Files Saved: 10


In [30]:
# ==========================================================
# Display the Final Phase 3 Summary
# This step summarizes the completed preprocessing process before moving to machine learning model development.
# ==========================================================

print("=" * 60)
print("PHASE 3: DATA PREPROCESSING COMPLETED")
print("=" * 60)

print("\nCleaned Dataset Rows:", len(df))
print("Original Input Features:", X.shape[1])
print("Processed Model Features:", X_train_processed.shape[1])

print("\nTraining Records:", X_train_processed.shape[0])
print("Testing Records:", X_test_processed.shape[0])

print("\nTraining Churn Cases:", int(y_train_processed["Churn"].sum()))
print("Testing Churn Cases:", int(y_test_processed["Churn"].sum()))

print("\nMissing Values in Training Data:",
      int(X_train_processed.isnull().sum().sum()))

print("Missing Values in Testing Data:",
      int(X_test_processed.isnull().sum().sum()))

print("\nPreprocessing Methods:")
print("- Continuous variables: StandardScaler")
print("- SeniorCitizen: Preserved as binary 0/1")
print("- Categorical variables: One-hot encoding")
print("- Target variable: No = 0, Yes = 1")
print("- Dataset split: 80% training and 20% testing")
print("- Stratification: Applied")
print("- Data leakage prevention: Applied")

print("\nOutput Directory:", output_directory)
print("\nThe processed data is ready for model development.")

PHASE 3: DATA PREPROCESSING COMPLETED

Cleaned Dataset Rows: 7032
Original Input Features: 19
Processed Model Features: 45

Training Records: 5625
Testing Records: 1407

Training Churn Cases: 1495
Testing Churn Cases: 374

Missing Values in Training Data: 0
Missing Values in Testing Data: 0

Preprocessing Methods:
- Continuous variables: StandardScaler
- SeniorCitizen: Preserved as binary 0/1
- Categorical variables: One-hot encoding
- Target variable: No = 0, Yes = 1
- Dataset split: 80% training and 20% testing
- Stratification: Applied
- Data leakage prevention: Applied

Output Directory: phase3_outputs

The processed data is ready for model development.


In [33]:
# ==========================================================
# Create and Download the Phase 3 Output Archive
# This step combines all processed files into one ZIP file so they can be stored safely outside Google Colab.
# ==========================================================

import shutil

from google.colab import files

archive_path = "Phase_3_Preprocessing_Outputs"

created_archive = shutil.make_archive(
    archive_path,
    "zip",
    output_directory
)

print("ZIP archive created successfully:")
print(created_archive)

files.download(created_archive)

ZIP archive created successfully:
/content/Phase_3_Preprocessing_Outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>